# ENTSO-E Data Analysis

This notebook provides exploratory data analysis (EDA) for electricity market data from the ENTSO-E Transparency Platform.

## Data Sources
- Day-ahead prices
- Actual load
- Generation by type

## Prerequisites
- Run the data scraper to collect data: `python ../data-scrapers/entsoe_scraper.py`
- Ensure data files are in the `../data-scrapers/data/` directory

In [ ]:
# Import required libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from datetime import datetime, timedelta

# Set plotting style
plt.style.use('seaborn-v0_8')
sns.set_palette('husl')
%matplotlib inline

# Set display options
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)

## 1. Load Data

Load the scraped data files from the data directory.

In [ ]:
# Define data directory
data_dir = Path('../data-scrapers/data')

# List available data files
if data_dir.exists():
    data_files = list(data_dir.glob('*.csv'))
    print(f"Found {len(data_files)} data files:")
    for file in data_files:
        print(f"  - {file.name}")
else:
    print(f"Data directory not found: {data_dir}")
    print("Please run the scraper first: python ../data-scrapers/entsoe_scraper.py")

In [ ]:
# Load day-ahead prices
def load_latest_file(pattern):
    """Load the most recent file matching the pattern."""
    files = sorted(data_dir.glob(pattern))
    if files:
        print(f"Loading: {files[-1].name}")
        df = pd.read_csv(files[-1], index_col=0, parse_dates=True)
        return df
    else:
        print(f"No files found matching: {pattern}")
        return None

# Load data
prices_df = load_latest_file('day_ahead_prices_*.csv')
load_df = load_latest_file('load_*.csv')
generation_df = load_latest_file('generation_*.csv')

## 2. Day-Ahead Price Analysis

In [ ]:
if prices_df is not None:
    print("Day-Ahead Prices Summary:")
    print(f"Period: {prices_df.index.min()} to {prices_df.index.max()}")
    print(f"\nStatistics (EUR/MWh):")
    print(prices_df.describe())
    
    # Plot price time series
    fig, ax = plt.subplots(figsize=(15, 6))
    prices_df.plot(ax=ax, linewidth=1.5)
    ax.set_title('Day-Ahead Electricity Prices', fontsize=16)
    ax.set_xlabel('Date', fontsize=12)
    ax.set_ylabel('Price (EUR/MWh)', fontsize=12)
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()
    
    # Price distribution
    fig, ax = plt.subplots(figsize=(10, 6))
    prices_df.hist(bins=50, ax=ax, edgecolor='black')
    ax.set_title('Price Distribution', fontsize=16)
    ax.set_xlabel('Price (EUR/MWh)', fontsize=12)
    ax.set_ylabel('Frequency', fontsize=12)
    plt.tight_layout()
    plt.show()
    
    # Daily patterns
    prices_df_copy = prices_df.copy()
    prices_df_copy['hour'] = prices_df_copy.index.hour
    prices_df_copy['day_of_week'] = prices_df_copy.index.day_name()
    
    fig, ax = plt.subplots(figsize=(12, 6))
    hourly_avg = prices_df_copy.groupby('hour').mean()
    hourly_avg.plot(kind='bar', ax=ax)
    ax.set_title('Average Price by Hour of Day', fontsize=16)
    ax.set_xlabel('Hour', fontsize=12)
    ax.set_ylabel('Average Price (EUR/MWh)', fontsize=12)
    plt.xticks(rotation=0)
    plt.tight_layout()
    plt.show()

## 3. Load Analysis

In [ ]:
if load_df is not None:
    print("Load Data Summary:")
    print(f"Period: {load_df.index.min()} to {load_df.index.max()}")
    print(f"\nStatistics (MW):")
    print(load_df.describe())
    
    # Plot load time series
    fig, ax = plt.subplots(figsize=(15, 6))
    load_df.plot(ax=ax, linewidth=1.5)
    ax.set_title('Actual Electricity Load', fontsize=16)
    ax.set_xlabel('Date', fontsize=12)
    ax.set_ylabel('Load (MW)', fontsize=12)
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()
    
    # Daily load patterns
    load_df_copy = load_df.copy()
    load_df_copy['hour'] = load_df_copy.index.hour
    load_df_copy['day_of_week'] = load_df_copy.index.day_name()
    
    fig, ax = plt.subplots(figsize=(12, 6))
    hourly_avg_load = load_df_copy.groupby('hour').mean()
    hourly_avg_load.plot(kind='bar', ax=ax, color='steelblue')
    ax.set_title('Average Load by Hour of Day', fontsize=16)
    ax.set_xlabel('Hour', fontsize=12)
    ax.set_ylabel('Average Load (MW)', fontsize=12)
    plt.xticks(rotation=0)
    plt.tight_layout()
    plt.show()

## 4. Generation Analysis

In [ ]:
if generation_df is not None:
    print("Generation Data Summary:")
    print(f"Period: {generation_df.index.min()} to {generation_df.index.max()}")
    print(f"\nGeneration Types: {list(generation_df.columns)}")
    print(f"\nStatistics (MW):")
    print(generation_df.describe())
    
    # Plot generation by type
    fig, ax = plt.subplots(figsize=(15, 8))
    generation_df.plot(ax=ax, linewidth=1.5)
    ax.set_title('Generation by Type', fontsize=16)
    ax.set_xlabel('Date', fontsize=12)
    ax.set_ylabel('Generation (MW)', fontsize=12)
    ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()
    
    # Generation mix (pie chart)
    fig, ax = plt.subplots(figsize=(10, 8))
    total_generation = generation_df.sum()
    total_generation.plot(kind='pie', ax=ax, autopct='%1.1f%%')
    ax.set_title('Generation Mix by Type', fontsize=16)
    ax.set_ylabel('')
    plt.tight_layout()
    plt.show()
    
    # Stacked area chart
    fig, ax = plt.subplots(figsize=(15, 8))
    generation_df.plot.area(ax=ax, alpha=0.7)
    ax.set_title('Generation Mix Over Time (Stacked)', fontsize=16)
    ax.set_xlabel('Date', fontsize=12)
    ax.set_ylabel('Generation (MW)', fontsize=12)
    ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
    plt.tight_layout()
    plt.show()

## 5. Combined Analysis: Price vs Load vs Renewable Generation

In [ ]:
if prices_df is not None and load_df is not None:
    # Combine data
    combined_df = pd.DataFrame({
        'price': prices_df.iloc[:, 0] if len(prices_df.columns) > 0 else prices_df,
        'load': load_df.iloc[:, 0] if len(load_df.columns) > 0 else load_df
    })
    
    # Calculate correlation
    correlation = combined_df.corr()
    print("Correlation between Price and Load:")
    print(correlation)
    
    # Scatter plot
    fig, ax = plt.subplots(figsize=(10, 6))
    ax.scatter(combined_df['load'], combined_df['price'], alpha=0.5)
    ax.set_title('Price vs Load', fontsize=16)
    ax.set_xlabel('Load (MW)', fontsize=12)
    ax.set_ylabel('Price (EUR/MWh)', fontsize=12)
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()
    
    # Dual axis plot
    fig, ax1 = plt.subplots(figsize=(15, 6))
    
    color = 'tab:blue'
    ax1.set_xlabel('Date', fontsize=12)
    ax1.set_ylabel('Load (MW)', color=color, fontsize=12)
    ax1.plot(combined_df.index, combined_df['load'], color=color, linewidth=1.5, label='Load')
    ax1.tick_params(axis='y', labelcolor=color)
    
    ax2 = ax1.twinx()
    color = 'tab:red'
    ax2.set_ylabel('Price (EUR/MWh)', color=color, fontsize=12)
    ax2.plot(combined_df.index, combined_df['price'], color=color, linewidth=1.5, label='Price')
    ax2.tick_params(axis='y', labelcolor=color)
    
    fig.suptitle('Price and Load Over Time', fontsize=16)
    fig.tight_layout()
    plt.show()

## 6. Export Analysis Results

In [ ]:
# Create summary report
summary = {}

if prices_df is not None:
    summary['price_stats'] = prices_df.describe().to_dict()
    summary['price_min'] = float(prices_df.min().min())
    summary['price_max'] = float(prices_df.max().max())
    summary['price_mean'] = float(prices_df.mean().mean())

if load_df is not None:
    summary['load_stats'] = load_df.describe().to_dict()
    summary['load_min'] = float(load_df.min().min())
    summary['load_max'] = float(load_df.max().max())
    summary['load_mean'] = float(load_df.mean().mean())

if generation_df is not None:
    summary['generation_mix'] = generation_df.sum().to_dict()

print("Analysis Summary:")
for key, value in summary.items():
    print(f"\n{key}:")
    print(value)

## Conclusions

Add your observations and insights here:
- Price patterns
- Load characteristics
- Generation mix insights
- Correlations and relationships